In [3]:
!pip install pymupdf pandas scikit-learn spacy joblib

In [4]:
import os
print(os.getcwd())

c:\Users\Shivam\Downloads


In [5]:
import fitz

In [6]:
!pip install pymupdf pandas numpy scikit-learn spacy joblib tqdm pdfplumber pytesseract pillow

  Using cached pdfplumber-0.11.9-py3-none-any.whl.metadata (43 kB)
  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
  Using cached pdfminer_six-20251230-py3-none-any.whl.metadata (4.3 kB)
  Using cached pypdfium2-5.6.0-py3-none-win_amd64.whl.metadata (68 kB)
Using cached pdfplumber-0.11.9-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20251230-py3-none-any.whl (6.6 MB)
Using cached pytesseract-0.3.13-py3-none-any.whl (14 kB)
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   -------------- ------------------------- 2.6/7.0 MB 29.7 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 27.3 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 32.1 MB/s  0:00:00
Using cached pypdfium2-5.6.0-py3-none-win_amd64.whl (3.7 MB)

   ---------------------------------------- 0/8 [pypdfium2]
   ---------------------------------------- 0/8 [pypdfium2]
   

In [7]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------- ------------------------------ 3.1/12.8 MB 16.4 MB/s eta 0:00:01
     ---------------------- ----------------- 7.1/12.8 MB 17.5 MB/s eta 0:00:01
     ------------------------------------ -- 12.1/12.8 MB 20.4 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 19.2 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [8]:
import os
import re
import fitz
import pandas as pd
import numpy as np
import joblib

from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

import spacy
nlp = spacy.load("en_core_web_sm")

In [9]:
import os
import fitz
import pandas as pd

dataset_path = r"C:\Users\Shivam\Downloads\dataset\dataset"

texts = []
labels = []

def extract_text(pdf_path):
    
    doc = fitz.open(pdf_path)
    text = ""
    
    for page in doc:
        text += page.get_text()
        
    return text


for folder in os.listdir(dataset_path):
    
    if folder == "Data to be Classified and Redacted":
        continue
        
    folder_path = os.path.join(dataset_path, folder)
    
    if os.path.isdir(folder_path):
        
        for file in os.listdir(folder_path):
            
            if file.endswith(".pdf"):
                
                pdf_path = os.path.join(folder_path, file)
                
                text = extract_text(pdf_path)
                
                texts.append(text)
                labels.append(folder)

print("Total training samples:", len(texts))

Total training samples: 294


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words="english"
)

X = vectorizer.fit_transform(texts)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (294, 3000)


In [11]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X, labels)

print("Model training completed")

Model training completed


In [12]:
import joblib

joblib.dump(model, "construction_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Model saved successfully")

Model saved successfully


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(classification_report(y_test, predictions))


                 precision    recall  f1-score   support

  Architectural       0.50      0.80      0.62        15
     Electrical       0.60      0.33      0.43         9
Fire Protection       1.00      0.70      0.82        10
     Mechanical       0.62      0.71      0.67         7
       Plumbing       0.69      0.69      0.69        13
     Structural       1.00      0.40      0.57         5

       accuracy                           0.64        59
      macro avg       0.74      0.61      0.63        59
   weighted avg       0.70      0.64      0.64        59



In [14]:
test_folder = r"C:\Users\Shivam\Downloads\dataset\dataset\Data to be Classified and Redacted"

results = []

for file in os.listdir(test_folder):
    
    if file.lower().endswith(".pdf"):
        
        pdf_path = os.path.join(test_folder, file)
        
        text = extract_text(pdf_path)
        
        X_test = vectorizer.transform([text])
        
        prediction = model.predict(X_test)[0]
        
        confidence = max(model.predict_proba(X_test)[0])
        
        results.append({
            "filename": file,
            "predicted_class": prediction,
            "confidence": confidence
        })

print("Total files classified:", len(results))

Total files classified: 50


In [15]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

email_pattern = r'\S+@\S+'
phone_pattern = r'\+?\d[\d -]{8,}\d'

def find_sensitive(text):
    
    sensitive = []
    
    # find emails
    emails = re.findall(email_pattern, text)
    
    # find phones
    phones = re.findall(phone_pattern, text)
    
    sensitive.extend(emails)
    sensitive.extend(phones)
    
    # detect names and organizations
    doc = nlp(text)
    
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE"]:
            sensitive.append(ent.text)
    
    return list(set(sensitive))

In [16]:
import re

email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
phone_pattern = r'\+?\d[\d\s\-]{8,}\d'

def find_sensitive(text):
    
    sensitive = []
    
    emails = re.findall(email_pattern, text)
    phones = re.findall(phone_pattern, text)
    
    sensitive.extend(emails)
    sensitive.extend(phones)
    
    doc = nlp(text)
    
    for ent in doc.ents:
        if ent.label_ in ["PERSON","ORG"]:
            sensitive.append(ent.text)
    
    return list(set(sensitive))

In [17]:
import os
import fitz
import re

test_folder = r"C:\Users\Shivam\Downloads\dataset\dataset\Data to be Classified and Redacted"

redacted_folder = "redacted_pdfs"
os.makedirs(redacted_folder, exist_ok=True)

email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
phone_pattern = r'\+?\d[\d\s\-]{8,}\d'

for file in os.listdir(test_folder):

    if file.lower().endswith(".pdf"):

        pdf_path = os.path.join(test_folder, file)
        doc = fitz.open(pdf_path)

        for page in doc:

            text = page.get_text()

            emails = re.findall(email_pattern, text)
            phones = re.findall(phone_pattern, text)

            sensitive = emails + phones

            for item in sensitive:

                areas = page.search_for(item)

                for area in areas:
                    page.add_redact_annot(area, fill=(0,0,0))

        for page in doc:
            page.apply_redactions()

        output_path = os.path.join(redacted_folder, file)
        doc.save(output_path)
        doc.close()

        print("Redacted:", file)

Redacted: 10 (2).pdf
Redacted: 10.pdf
Redacted: 103.pdf
Redacted: 107.pdf
Redacted: 12.pdf
Redacted: 124.pdf
Redacted: 141.pdf
Redacted: 19 (2).pdf
Redacted: 19.pdf
Redacted: 23.pdf
Redacted: 25R1 OHT Details 26.03.21-OHT.pdf
Redacted: 26.pdf
Redacted: 30 (2).pdf
Redacted: 30.pdf
Redacted: 30R5 Toilet Details 02.04.21-PLUMBING DETAILS.pdf
Redacted: 38.pdf
Redacted: 4.pdf
Redacted: 40 Security Cabin-Security Cabin.pdf
Redacted: 40.pdf
Redacted: 46.pdf
Redacted: 5.pdf
Redacted: 54.pdf
Redacted: 58.pdf
Redacted: 67.pdf
Redacted: 8.pdf
Redacted: a-Rinker_021.pdf
Redacted: a-Rinker_023.pdf
Redacted: a-Rinker_041.pdf
Redacted: A-sample-drawing-package-1and2family.pdf
Redacted: A1.2 - DEMOLITION PLAN SHEETS.pdf
Redacted: E4.1 - ELECTRICAL LOW VOLTAGE NEW WORK PLANS.pdf
Redacted: electricalplan.pdf
Redacted: fp11.pdf
Redacted: fp6.pdf
Redacted: M-1 Mechanical Plan--.pdf
Redacted: m-Rinker_062.pdf
Redacted: m-Rinker_067.pdf
Redacted: m2.pdf
Redacted: P-NGS-riser-phskc-example-plans.pdf
Redacted

In [18]:
import os
import fitz

dataset_path = r"C:\Users\Shivam\Downloads\dataset\dataset"

texts = []
labels = []

classes = [
    "Architectural",
    "Structural",
    "Mechanical",
    "Plumbing",
    "Electrical",
    "Fire Protection"
]

for label in classes:

    folder = os.path.join(dataset_path, label)

    for file in os.listdir(folder):

        if file.lower().endswith(".pdf"):

            pdf_path = os.path.join(folder, file)

            doc = fitz.open(pdf_path)

            text = ""

            for page in doc:
                text += page.get_text()

            texts.append(text)
            labels.append(label)

print("Total samples:", len(texts))

Total samples: 294


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = vectorizer.fit_transform(texts)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (294, 5000)


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [21]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)

model.fit(X_train, y_train)

print("Training completed")

Training completed


In [22]:
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print(classification_report(y_test, predictions))

Accuracy: 0.711864406779661
                 precision    recall  f1-score   support

  Architectural       0.48      0.79      0.59        14
     Electrical       0.67      0.22      0.33         9
Fire Protection       1.00      1.00      1.00         9
     Mechanical       0.67      0.60      0.63        10
       Plumbing       0.92      0.86      0.89        14
     Structural       1.00      0.67      0.80         3

       accuracy                           0.71        59
      macro avg       0.79      0.69      0.71        59
   weighted avg       0.75      0.71      0.70        59



In [23]:
import re

def detect_sheet_code(text):

    if re.search(r'\bA\d{2,3}\b', text):
        return "Architectural"

    if re.search(r'\bS\d{2,3}\b', text):
        return "Structural"

    if re.search(r'\bM\d{2,3}\b', text):
        return "Mechanical"

    if re.search(r'\bP\d{2,3}\b', text):
        return "Plumbing"

    if re.search(r'\bE\d{2,3}\b', text):
        return "Electrical"

    if re.search(r'\bFP\d{2,3}\b', text):
        return "Fire Protection"

    return None

In [24]:
code = detect_sheet_code(text)

if code:
    prediction = code
else:
    prediction = model.predict(X_test)[0]

predictions = model.predict(X_test)

In [25]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.711864406779661


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=8000,
    ngram_range=(1,2)   # unigram + bigram
)

X = vectorizer.fit_transform(texts)

print(X.shape)

(294, 8000)


In [27]:
def keyword_features(text):

    keywords = {
        "Architectural": ["floor","elevation","door","window"],
        "Structural": ["beam","column","rebar"],
        "Mechanical": ["duct","hvac","ventilation"],
        "Plumbing": ["pipe","drain","fixture","plumbed"],
        "Electrical": ["panel","lighting","wiring"],
        "Fire Protection": ["sprinkler","alarm","hydrant"]
    }

    counts = []

    text = text.lower()

    for cls in keywords:
        count = sum(text.count(word) for word in keywords[cls])
        counts.append(count)

    return counts

In [28]:
import numpy as np

keyword_matrix = np.array([keyword_features(t) for t in texts])

In [29]:
from scipy.sparse import hstack

X_combined = hstack([X, keyword_matrix])

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [31]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [32]:
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("New Accuracy:", accuracy)

print(classification_report(y_test, predictions))

New Accuracy: 0.711864406779661
                 precision    recall  f1-score   support

  Architectural       0.50      0.79      0.61        14
     Electrical       0.67      0.22      0.33         9
Fire Protection       1.00      1.00      1.00         9
     Mechanical       0.67      0.60      0.63        10
       Plumbing       0.86      0.86      0.86        14
     Structural       1.00      0.67      0.80         3

       accuracy                           0.71        59
      macro avg       0.78      0.69      0.71        59
   weighted avg       0.74      0.71      0.70        59



In [33]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X,
    labels,
    cv=cv,
    scoring="accuracy"
)

print("Scores:", scores)
print("Average accuracy:", scores.mean())

Scores: [0.71186441 0.71186441 0.62711864 0.72881356 0.75862069]
Average accuracy: 0.707656341320865


In [34]:
from sklearn.linear_model import LogisticRegression
final_model = LogisticRegression(max_iter=20000)

final_model.fit(X, labels)

print("Final model trained on full dataset")

Final model trained on full dataset


In [35]:
import joblib

joblib.dump(final_model, "construction_classifier.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model saved successfully")

Model saved successfully


In [36]:
import fitz   # PyMuPDF

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    
    text = ""
    
    for page in doc:
        text += page.get_text()
        
    doc.close()
    
    return text

In [37]:
import os

folder = r"C:\Users\Shivam\Downloads\dataset\dataset\Architectural"

files = os.listdir(folder)

pdf_file = [f for f in files if f.lower().endswith(".pdf")][0]

sample_pdf = os.path.join(folder, pdf_file)

print("Using file:", sample_pdf)

text = extract_text(sample_pdf)

print(text[:500])


Using file: C:\Users\Shivam\Downloads\dataset\dataset\Architectural\1.pdf
WOOD PLANK BENCH
WOOD PLANK BENCH
SOFA
SINK
LOCKER
MODULAR SINGLE BED
W/ DESK
TELEVISION
WC.
F.D.
DESK
ISOLATION RM 3. w/ T&B
AREA: 12.0sq.mtrs.
T&B
ACU
FLOATING SHELF
SOFA
SINK
LOCKER
MODULAR SINGLE BED
W/ DESK
TELEVISION
SHO.
WC.
F.D.
DESK
ISOLATION RM. 2 w/ T&B
AREA: 12.0sq.mtrs.
T&B
ACU
FLOATING SHELF
SOFA
SINK
LOCKER
MODULAR SINGLE BED
W/ DESK
TELEVISION
WC.
F.D.
DESK
ISOLATION RM. 1 w/ T&B
AREA: 12.0sq.mtrs.
T&B
ACU
FLOATING SHELF
NICHE
NICHE
NICHE
SHO.
SHO.
WOOD PLANK BENCH
VERANDA
EL. + 


In [38]:
model.fit(X, labels)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [40]:
import os
import fitz
import pandas as pd
import joblib

# -----------------------------
# Load trained model and vectorizer
# -----------------------------

model = joblib.load("construction_classifier.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

# -----------------------------
# Path to test data
# -----------------------------

test_folder = r"C:\Users\Shivam\Downloads\dataset\dataset\Data to be Classified and Redacted"

# -----------------------------
# Function to extract text
# -----------------------------

def extract_text(pdf_path):
    
    doc = fitz.open(pdf_path)
    text = ""
    
    for page in doc:
        text += page.get_text()
        
    doc.close()
    
    return text

# -----------------------------
# Run predictions
# -----------------------------

results = []

for file in os.listdir(test_folder):
    
    if file.lower().endswith(".pdf"):
        
        pdf_path = os.path.join(test_folder, file)
        
        text = extract_text(pdf_path)
        
        X_test = vectorizer.transform([text])
        
        prediction = model.predict(X_test)[0]
        
        confidence = model.predict_proba(X_test).max()
        
        results.append({
            "filename": file,
            "predicted_class": prediction,
            "confidence": confidence
        })

# -----------------------------
# Save results to CSV
# -----------------------------

df = pd.DataFrame(results)

df.to_csv("classification_output.csv", index=False)

print("Classification completed")

max_len = 32

df_sorted = df.sort_values(by="confidence", ascending=False)

print(f"{'File Name':<34} {'Class':<20} {'Confidence':<10}")
print("-"*70)

for _, row in df_sorted.iterrows():
    
    name = row['filename']
    
    if len(name) > max_len:
        name = name[:max_len-3] + "..."
        
    print(f"{name:<34} {row['predicted_class']:<20} {row['confidence']:.2f}")

Classification completed
File Name                          Class                Confidence
----------------------------------------------------------------------
P-NIST_Reference_Building_Plu...   Plumbing             0.94
23.pdf                             Mechanical           0.86
19.pdf                             Mechanical           0.82
58.pdf                             Plumbing             0.75
5.pdf                              Fire Protection      0.70
fp6.pdf                            Fire Protection      0.70
4.pdf                              Plumbing             0.67
[3] 64x60 house plan with4 be...   Architectural        0.66
30 (2).pdf                         Plumbing             0.65
10.pdf                             Mechanical           0.64
40.pdf                             Plumbing             0.64
30.pdf                             Mechanical           0.63
124.pdf                            Architectural        0.63
Sheets_ FA-102.00 - CONVERTER...   Fire Prot